In [1]:
import geopandas as gpd
import pandas as pd
from glob import glob
import s3fs
from datetime import datetime
import folium

In [2]:
# for connecting later
s3 = s3fs.S3FileSystem(anon=False)

In [3]:
current_yr = datetime.now().year # generalize to arbitrary point in time

# largefire files for that year
af_files = s3.glob(f's3://maap-ops-workspace/shared/gsfc_landslides/FEDSoutput-v3/CONUS/{current_yr}/*.parq')
snapshot_files = s3.glob(f's3://maap-ops-workspace/shared/gsfc_landslides/FEDSoutput-v3/CONUS/{current_yr}/Snapshot/**/perimeter.fgb')

In [4]:
# -1 should be most recent file due to file naming conventions
af_recent_ts = gpd.read_parquet('s3://'+af_files[-1])
snapshot_today = gpd.read_file('s3://'+snapshot_files[-1])
# af_recent_ts.set_index('fireID',inplace=True)

In [7]:
def allfires_nifc_data_join(allfires_gdf,active_only=True):
    if active_only:
        nifc_perimeters = gpd.read_file('https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Interagency_Perimeters_Current/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson')
    else:
        nifc_perimeters = gpd.read_file('https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Interagency_Perimeters_YearToDate/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson')

    # define geometry for allfires spatial join
    allfires_gdf.set_geometry('hull',inplace=True)

    
    return allfires_gdf

In [11]:
af_recent_ts_nifc = allfires_nifc_data_join(af_recent_ts,active_only=True)
af_recent_ts_nifc

,,mergeid,invalid,ftype,n_pixels,n_newpixels,farea,fperim,flinelen,duration,pixden,meanFRP,t_st,t_ed,hull,fline,nfp
fireID,t,,,,,,,,,,,,,,,,
1,2025-01-01,1,False,0,1,1,0.141000,1.177624,0.000000,0.0,7.092199,0.740000,2025-01-01,2025-01-01,"POLYGON ((1978981.359 412260.116, 1978980.456 ...",None,MULTIPOINT ((1978793.859 412260.116))
2,2025-01-01,2,False,0,2,2,0.220539,2.355248,2.355248,0.0,9.068709,0.895000,2025-01-01,2025-01-01,"MULTIPOLYGON (((1973150.386 -600091.495, 19731...","MULTILINESTRING ((1973150.386 -600091.495, 197...","MULTIPOINT ((1972578.53 -600123.697), (1972962..."
3,2025-01-01,3,False,0,1,1,0.141000,1.177624,0.000000,0.0,7.092199,0.660000,2025-01-01,2025-01-01,"POLYGON ((2016887.763 -545913.311, 2016886.86 ...",None,MULTIPOINT ((2016700.263 -545913.311))
4,2025-01-01,4,False,2,1,1,0.141000,1.177624,0.000000,0.0,7.092199,0.670000,2025-01-01,2025-01-01,"POLYGON ((2034952.767 -514173.019, 2034951.864...",None,MULTIPOINT ((2034765.267 -514173.019))
5,2025-01-01,5,False,0,1,1,0.141000,1.177624,0.000000,0.0,7.092199,0.330000,2025-01-01,2025-01-01,"POLYGON ((2094762.735 -507147.968, 2094761.833...",None,MULTIPOINT ((2094575.235 -507147.968))
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51917,2025-05-20,51917,False,0,1,1,0.141000,1.177624,1.177624,0.0,7.092199,0.780000,2025-05-20,2025-05-20,"POLYGON ((-353229.242 -2140578.127, -353230.14...","LINESTRING (-353229.242 -2140578.127, -353230....",MULTIPOINT ((-353416.742 -2140578.127))
51918,2025-05-20,51918,False,0,26,26,4.985689,10.734624,10.419745,0.0,5.214926,9.969231,2025-05-20,2025-05-20,"POLYGON ((-106203.884 -2199449.021, -106203.82...","MULTILINESTRING ((-106203.884 -2199449.021, -1...","MULTIPOINT ((-109187.989 -2196391.183), (-1087..."
51919,2025-05-20,51919,False,0,3,3,0.702036,4.289646,4.289646,0.0,4.273285,0.990000,2025-05-20,2025-05-20,"POLYGON ((-105187.358 -2200895.371, -105188.63...","LINESTRING (-105187.358 -2200895.371, -105188....","MULTIPOINT ((-105000.012 -2199329.202), (-1052..."


In [54]:
snapshot_today.set_index('fireID',inplace=True)

In [55]:
# # keep only most recent perimeter observations for matching with NIFC
# lf_today = lf_recent_ts.sort_values(by=['farea'], ascending=False).drop_duplicates(subset='mergeid', keep='first')

In [56]:
# # make sure there's only one fire ID per entry
# assert lf_today.index.nunique()==len(lf_today), 'Fire IDs must be unique!'

In [57]:
# load in NIFC data, can use either one depending on use case

# all 2025 perimeters
nifc_all = gpd.read_file('https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Interagency_Perimeters_YearToDate/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson')
nifc_all = nifc_all.to_crs(lf_today.crs) # reproject

# only active perimeters
#  nifc_recent = gpd.read_file('https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Interagency_Perimeters_Current/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson')

In [58]:
# convert discovery date col to datetime
nifc_all['attr_FireDiscoveryDateTime'] = pd.to_datetime(nifc_all['attr_FireDiscoveryDateTime'],unit='ms')

# clean up irwin id
nifc_all['poly_IRWINID'] = nifc_all['poly_IRWINID'].apply(lambda x: x.strip('{}'))

In [59]:
nifc_all['attr_IncidentTypeCategory'].unique()

array(['WF', 'RX'], dtype=object)

In [60]:
print(f'All NIFC fires: {nifc_all.shape[0]}')

All NIFC fires: 2000


In [62]:
# perform spatial join between NIFC and FEDS 
sjoin = snapshot_today.sjoin(nifc_all)
sjoin['feds_nifc_time_diff'] = abs(sjoin['t_st'] - sjoin['attr_FireDiscoveryDateTime']) / pd.to_timedelta('24h')

# optional time filter to make sure only those with near concurrent ignition dates are preserved
# sjoin = sjoin[sjoin['feds_nifc_time_diff']<=10] 

# join back to AF data, only keep matches
snapshot_today_merged = snapshot_today.merge(sjoin[['poly_IncidentName','poly_IRWINID','attr_FireDiscoveryDateTime','attr_IncidentTypeCategory']],left_index=True,right_index=True)

In [63]:
print(f'Fire IDs with matches: {snapshot_today_merged.index.nunique()}')
print(f'NIFC events with matches: {snapshot_today_merged['poly_IRWINID'].nunique()}')

Fire IDs with matches: 38
NIFC events with matches: 46


In [ ]:
m = nifc_all.sort_values('attr_FireDiscoveryDateTime',ascending=False)[['geometry','attr_FireDiscoveryDateTime','poly_IncidentName',]].explore(name='NIFC Perimeters',color='black')
snapshot_today_merged[['geometry','attr_FireDiscoveryDateTime','t_st','t_ed','poly_IncidentName','poly_IRWINID']].explore(m=m,name='FEDS-NIFC Matched Perimeters',color='red')
folium.LayerControl().add_to(m)
m

In [66]:
nifc_records_to_merge = snapshot_today_merged[['attr_FireDiscoveryDateTime','poly_IncidentName','poly_IRWINID']].copy()

In [67]:
# aggregate all unique NIFC for each fireID
grouped_records = nifc_records_to_merge.groupby('fireID')[nifc_records_to_merge.columns].agg(['unique'])

grouped_records = grouped_records.droplevel(level=1,axis=1) # clean up columns from agg operation

grouped_records = grouped_records.rename(columns={'attr_FireDiscoveryDateTime': 'NIFC_DiscoveryDT', 
                                                              'poly_IncidentName': 'NIFC_IncidentName',
                                                              'poly_IRWINID': 'NIFC_IRWINID',
                                                              'attr_IncidentTypeCategory': 'NIFC_IncidentType'})

# clear list (array) instance if only single entry per fire
for col in grouped_records.columns:
    grouped_records[col] = grouped_records[col].apply(lambda x: x[0] if len(x) == 1 else list(x))

In [89]:
grouped_records.columns

Index(['NIFC_DiscoveryDT', 'NIFC_IncidentName', 'NIFC_IRWINID'], dtype='object')

In [69]:
snapshot_today_w_nifc = snapshot_today.merge(grouped_records,left_index=True,right_index=True,how='left')

In [85]:
# snapshot_today_w_nifc.to_file('~/sample_snapshot.fgb')

In [ ]:
sn.explore()

In [ ]:
m = nifc_all.sort_values('attr_FireDiscoveryDateTime',ascending=False)[['geometry','attr_FireDiscoveryDateTime','poly_IncidentName',]].explore(name='NIFC Perimeters',color='black')
lf_recent_ts_w_nifc.sort_values('t').drop_duplicates(subset='mergeid',keep='last')[['geometry','t_st','NIFC_DiscoveryDT', 'NIFC_IncidentName', 'NIFC_IRWINID']].explore(m=m,name='FEDS-NIFC Matched Perimeters',color='red')
folium.LayerControl().add_to(m)
m

In [21]:
nc_wildfire = lf_recent_ts_w_nifc.loc[28074].sort_values('t',ascending=False)
nifc_nc_wildfire = nifc_all.sjoin(nc_wildfire)
m = nifc_nc_wildfire[['geometry','attr_FireDiscoveryDateTime','poly_IncidentName','poly_IRWINID']].explore(name='NIFC NC Perimeters',
                                                                                                           style_kwds={'fillOpacity':0.1})
nc_wildfire[['geometry','NIFC_Discovery_DT','t', 'NIFC_IncidentName', 'NIFC_IRWINID']].explore(m=m,style_kwds={'fillOpacity':0.1},
                                                                                               column='t',cmap='viridis',
                                                                                               name='FEDS NIFC Perimeters')
folium.LayerControl().add_to(m)
m

KeyError: "['NIFC_Discovery_DT'] not in index"